# Scikit-Learn Fundamentals: The Grammar of Machine Learning

## Why Scikit-Learn?

Imagine you're building trading strategies at a hedge fund. You need to:
- Predict next month's stock returns using dozens of factors
- Classify whether a bond will default
- Detect anomalous trading patterns

Each of these requires different models, different preprocessing, different evaluation. Without a consistent framework, you'd waste hours rewriting boilerplate code.

**Scikit-learn solves this.** It provides a unified API where every model, every transformer, every evaluation tool works the same way. Learn the pattern once, apply it everywhere.

By the end of this notebook, you'll understand:
- How sklearn is designed (and why it's designed that way)
- The three core interfaces: Estimator, Transformer, Predictor
- How to chain operations with Pipelines
- **Critical**: How to avoid data leakage (the #1 mistake in ML for finance)

Let's get started.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Sklearn imports - we'll explain each as we use them
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# For reproducibility
np.random.seed(42)

# Plotting defaults
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("Ready to learn sklearn!")

## Part 1: The Machine Learning Workflow

Before diving into sklearn, let's understand the typical ML workflow. Every ML project follows roughly the same steps:

```
1. INGEST      →  Load and explore your data
2. TRANSFORM   →  Clean, scale, engineer features  
3. SPLIT       →  Separate training and test data
4. TRAIN       →  Fit a model on training data
5. PREDICT     →  Make predictions on test data
6. EVALUATE    →  Measure how well the model performs
7. ITERATE     →  Adjust and try again
```

Sklearn is designed to make steps 2-6 seamless. Let's see this in action with a finance example.

### Our Problem: Factor Model for Stock Returns

We'll build a simple factor model - a workhorse of quantitative finance. The idea: stock returns can be explained by exposure to common factors.

$$R_i = \alpha + \beta_1 \cdot \text{Market} + \beta_2 \cdot \text{Size} + \beta_3 \cdot \text{Value} + \epsilon$$

Let's generate some realistic data:

In [ ]:
def generate_factor_data(n_samples=1000, n_stocks=50, seed=42):
    """
    Generate synthetic factor model data.
    
    We simulate:
    - Market factor (MKT): Overall market movement
    - Size factor (SMB): Small minus Big
    - Value factor (HML): High minus Low book-to-market
    - Momentum factor (MOM): Past winners minus losers
    
    Each stock has factor exposures (betas) and an idiosyncratic component.
    """
    np.random.seed(seed)
    
    # Generate factor returns (what the factors returned each period)
    factors = pd.DataFrame({
        'MKT': np.random.normal(0.0005, 0.02, n_samples),   # ~12% annual, 20% vol
        'SMB': np.random.normal(0.0002, 0.01, n_samples),   # Small size premium
        'HML': np.random.normal(0.0003, 0.012, n_samples),  # Value premium
        'MOM': np.random.normal(0.0004, 0.015, n_samples),  # Momentum
    })
    
    # Generate stock-specific factor loadings (betas)
    # In reality, these would be estimated from historical data
    betas = pd.DataFrame({
        'MKT': np.random.uniform(0.5, 1.5, n_stocks),       # Market beta
        'SMB': np.random.uniform(-0.5, 1.0, n_stocks),      # Size exposure
        'HML': np.random.uniform(-0.5, 0.8, n_stocks),      # Value exposure  
        'MOM': np.random.uniform(-0.3, 0.5, n_stocks),      # Momentum exposure
    })
    
    # Generate stock returns: R = sum(beta * factor) + idiosyncratic
    returns_list = []
    for stock_idx in range(n_stocks):
        stock_betas = betas.iloc[stock_idx]
        systematic_return = (factors * stock_betas).sum(axis=1)
        idiosyncratic = np.random.normal(0, 0.015, n_samples)  # Stock-specific noise
        stock_return = systematic_return + idiosyncratic
        returns_list.append(stock_return)
    
    # Create a panel dataset
    data = []
    for t in range(n_samples):
        for s in range(n_stocks):
            data.append({
                'date': t,
                'stock_id': s,
                'return': returns_list[s].iloc[t],
                'MKT': factors['MKT'].iloc[t],
                'SMB': factors['SMB'].iloc[t],
                'HML': factors['HML'].iloc[t],
                'MOM': factors['MOM'].iloc[t],
                'true_beta_MKT': betas['MKT'].iloc[s],  # We'll hide these later
                'true_beta_SMB': betas['SMB'].iloc[s],
                'true_beta_HML': betas['HML'].iloc[s],
                'true_beta_MOM': betas['MOM'].iloc[s],
            })
    
    return pd.DataFrame(data)

# Generate our dataset
df = generate_factor_data(n_samples=500, n_stocks=30)
print(f"Generated {len(df):,} observations")
print(f"Date range: 0 to {df['date'].max()}")
print(f"Number of stocks: {df['stock_id'].nunique()}")
df.head(10)

Now let's simplify: we'll work with aggregated data where we try to predict returns using factor exposures.

In [ ]:
# For this example, let's work with a simpler cross-sectional dataset
# We'll create features that might predict next-period returns

np.random.seed(42)
n_samples = 1000

# Features that might predict returns (in practice, these would be real data)
X = pd.DataFrame({
    'market_cap_log': np.random.normal(10, 2, n_samples),           # Log market cap
    'book_to_market': np.random.exponential(0.5, n_samples),         # B/M ratio
    'momentum_12m': np.random.normal(0.1, 0.3, n_samples),           # Past 12m return
    'volatility_60d': np.random.exponential(0.02, n_samples) + 0.01, # 60-day vol
    'pe_ratio': np.random.lognormal(2.5, 0.5, n_samples),            # P/E ratio
})

# Generate returns with some true relationship to features
# In reality, these relationships are much weaker and noisier!
true_coefficients = np.array([-0.001, 0.015, 0.02, -0.5, -0.0001])
y = X.values @ true_coefficients + np.random.normal(0, 0.03, n_samples)
y = pd.Series(y, name='next_month_return')

print("Features (X):")
print(X.describe().round(3))
print(f"\nTarget (y) - Next Month Return:")
print(f"Mean: {y.mean():.4f}, Std: {y.std():.4f}")

---

## Part 2: Sklearn's Three Core Interfaces

Sklearn is built around three simple interfaces. Every tool in the library implements one or more of these:

| Interface | Key Method | What It Does | Example |
|-----------|------------|--------------|----------|
| **Estimator** | `fit(X, y)` | Learns from data | Every model, every scaler |
| **Predictor** | `predict(X)` | Makes predictions | LinearRegression, RandomForest |
| **Transformer** | `transform(X)` | Transforms data | StandardScaler, PCA |

The beauty is that **all** sklearn objects follow this pattern. Once you learn it, you can use any model or transformer without reading new documentation.

### 2.1 Estimators: The `fit()` Method

An **Estimator** is anything that learns from data. The `fit(X, y)` method takes:
- `X`: Features (a 2D array or DataFrame)
- `y`: Target (optional for some estimators like scalers)

After fitting, the estimator stores what it learned as attributes (usually ending with `_`).

In [ ]:
# LinearRegression is an Estimator
model = LinearRegression()

# Before fitting - no learned parameters
print("Before fit():")
print(f"  Has coefficients? {hasattr(model, 'coef_')}")

# Fit the model
model.fit(X, y)

# After fitting - parameters are learned!
print("\nAfter fit():")
print(f"  Coefficients: {model.coef_.round(4)}")
print(f"  Intercept: {model.intercept_:.4f}")
print(f"  Feature names: {list(X.columns)}")

In [ ]:
# Compare learned coefficients to true coefficients
comparison = pd.DataFrame({
    'Feature': X.columns,
    'True Coefficient': true_coefficients,
    'Learned Coefficient': model.coef_
})
comparison['Difference'] = comparison['Learned Coefficient'] - comparison['True Coefficient']
print("Coefficient Comparison:")
print(comparison.to_string(index=False))
print("\nNot bad! The model recovered the true relationships fairly well.")

### 2.2 Predictors: The `predict()` Method

A **Predictor** makes predictions on new data. After fitting, call `predict(X_new)` to get predictions.

In [ ]:
# Make predictions on the same data (just for demonstration)
y_pred = model.predict(X)

print(f"Predictions shape: {y_pred.shape}")
print(f"\nFirst 10 predictions vs actuals:")
comparison_df = pd.DataFrame({
    'Actual': y[:10].values,
    'Predicted': y_pred[:10],
    'Error': (y[:10].values - y_pred[:10])
})
print(comparison_df.round(4).to_string(index=False))

In [ ]:
# Visualize predictions vs actuals
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
axes[0].scatter(y, y_pred, alpha=0.3, edgecolors='none')
axes[0].plot([y.min(), y.max()], [y.min(), y.max()], 'r--', linewidth=2, label='Perfect fit')
axes[0].set_xlabel('Actual Return')
axes[0].set_ylabel('Predicted Return')
axes[0].set_title('Predicted vs Actual Returns')
axes[0].legend()

# Residual plot
residuals = y - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.3, edgecolors='none')
axes[1].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[1].set_xlabel('Predicted Return')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residual Plot (should be random around 0)')

plt.tight_layout()

### 2.3 Transformers: The `transform()` Method

A **Transformer** modifies data. Common uses:
- Scaling features (StandardScaler, MinMaxScaler)
- Encoding categories (OneHotEncoder)
- Dimensionality reduction (PCA)

Transformers also have `fit()` because they need to learn parameters (like mean and std for scaling).

In [ ]:
# StandardScaler is a Transformer
scaler = StandardScaler()

# Before fitting
print("Before fit():")
print(f"  Has mean? {hasattr(scaler, 'mean_')}")

# Fit - learns mean and std from data
scaler.fit(X)

print("\nAfter fit():")
print(f"  Learned means: {scaler.mean_.round(3)}")
print(f"  Learned stds: {scaler.scale_.round(3)}")

In [ ]:
# Transform - apply the learned scaling
X_scaled = scaler.transform(X)

# The result is a numpy array
print(f"Type of X_scaled: {type(X_scaled)}")
print(f"Shape: {X_scaled.shape}")

# Verify it's standardized (mean ≈ 0, std ≈ 1)
print(f"\nAfter scaling:")
print(f"  Column means: {X_scaled.mean(axis=0).round(4)}")
print(f"  Column stds: {X_scaled.std(axis=0).round(4)}")

In [ ]:
# Common shortcut: fit_transform() does both in one step
scaler2 = StandardScaler()
X_scaled2 = scaler2.fit_transform(X)

# Verify they're the same
print(f"Are they identical? {np.allclose(X_scaled, X_scaled2)}")

### Quick Check: Why Does This Design Matter?

The consistent interface means you can swap components easily:

```python
# Want to try a different model? Just change one line:
model = LinearRegression()  # or...
model = Ridge(alpha=1.0)    # or...
model = Lasso(alpha=0.1)    # or...
model = RandomForestRegressor(n_estimators=100)

# The rest of your code stays the same!
model.fit(X_train, y_train)
predictions = model.predict(X_test)
```

In [ ]:
# Let's prove it - same code, three different models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge (alpha=1.0)': Ridge(alpha=1.0),
    'Lasso (alpha=0.001)': Lasso(alpha=0.001),
}

# Split data first (we'll learn more about this later)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

results = []
for name, model in models.items():
    # Same code for every model!
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    results.append({'Model': name, 'MSE': mse, 'R²': r2})

results_df = pd.DataFrame(results)
print("Model Comparison:")
print(results_df.round(6).to_string(index=False))

---

## Part 3: Pipelines - Chaining Operations Together

In practice, ML workflows involve multiple steps:
1. Scale the features
2. Maybe reduce dimensions
3. Train a model

**Pipelines** let you chain these steps into a single object that behaves like one estimator.

### 3.1 Why Pipelines?

Without pipelines, you might write code like this:

In [ ]:
# The "manual" way (DON'T DO THIS in production)

# Step 1: Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # Important: transform, not fit_transform!

# Step 2: Train model
model = Ridge(alpha=1.0)
model.fit(X_train_scaled, y_train)

# Step 3: Predict
y_pred = model.predict(X_test_scaled)

print(f"MSE: {mean_squared_error(y_test, y_pred):.6f}")

**Problems with the manual approach:**
1. Easy to make mistakes (using `fit_transform` on test data = data leakage!)
2. Multiple objects to track
3. Harder to save/load
4. Doesn't work with cross-validation

### 3.2 Building a Pipeline

A Pipeline is a list of `(name, estimator)` tuples. The last step can be any estimator; all previous steps must be transformers.

In [ ]:
from sklearn.pipeline import Pipeline

# Create a pipeline
pipe = Pipeline([
    ('scaler', StandardScaler()),     # Step 1: Scale
    ('model', Ridge(alpha=1.0))       # Step 2: Model
])

# The pipeline behaves like a single estimator!
print("Pipeline:")
print(pipe)

In [ ]:
# Fit the pipeline - it automatically handles the chain
pipe.fit(X_train, y_train)

# Predict - it automatically scales then predicts
y_pred_pipe = pipe.predict(X_test)

print(f"MSE (pipeline): {mean_squared_error(y_test, y_pred_pipe):.6f}")
print(f"Same as manual? {np.allclose(y_pred, y_pred_pipe)}")

### 3.3 Accessing Pipeline Components

You can access individual steps using `named_steps` or indexing:

In [ ]:
# Access by name
print("Scaler means:")
print(pipe.named_steps['scaler'].mean_.round(3))

print("\nModel coefficients:")
print(pipe.named_steps['model'].coef_.round(4))

# Access by index
print(f"\nFirst step: {pipe[0]}")
print(f"Last step: {pipe[-1]}")

### 3.4 Shorthand with `make_pipeline`

If you don't need custom names, use `make_pipeline` for cleaner code:

In [ ]:
from sklearn.pipeline import make_pipeline

# Automatic naming based on class names
pipe_auto = make_pipeline(
    StandardScaler(),
    Ridge(alpha=1.0)
)

print("Auto-named pipeline:")
print(pipe_auto)

# Access by auto-generated lowercase names
pipe_auto.fit(X_train, y_train)
print(f"\nAccess via auto name: {pipe_auto.named_steps['ridge']}")

---

## Part 4: Data Leakage - The Silent Killer

**Data leakage** occurs when information from the test set "leaks" into the training process. This is the #1 cause of overly optimistic backtests in quantitative finance.

### Common Sources of Leakage:

1. **Using future data in features**: Creating a "lagged" feature that accidentally includes the target
2. **Scaling before splitting**: Computing mean/std on all data, including test
3. **Shuffling time series**: Using tomorrow's data to help predict today

Let's see a dramatic example of how leakage can make a worthless model look amazing:

### Example: The "Amazing" Model That's Actually Useless

Imagine you're building a model to predict tomorrow's stock return. You engineer some features, train a model, and get an R² of 0.85. Amazing!

But wait... you accidentally included future information in your features. Let's see what happens:

In [ ]:
# Generate a realistic time series of stock returns
np.random.seed(42)
n_days = 500

# Daily returns: random walk with slight drift (realistic - hard to predict!)
returns = np.random.normal(0.0003, 0.02, n_days)  # ~7.5% annual return, 20% vol

# Create a DataFrame with dates
dates = pd.date_range(end='2024-12-31', periods=n_days, freq='B')
df_ts = pd.DataFrame({'return': returns}, index=dates)

# Target: next day's return (what we're trying to predict)
df_ts['target'] = df_ts['return'].shift(-1)

print(f"Dataset: {len(df_ts)} trading days")
print(f"Target (next-day return): mean={df_ts['target'].mean():.4f}, std={df_ts['target'].std():.4f}")
print("\nIn reality, next-day returns are nearly impossible to predict...")
df_ts.head()

In [ ]:
# ============================================================
# THE CORRECT WAY: Use only past data to predict the future
# ============================================================

# Create legitimate features (past data only)
df_correct = df_ts.copy()
df_correct['return_lag1'] = df_correct['return'].shift(1)   # Yesterday's return
df_correct['return_lag2'] = df_correct['return'].shift(2)   # 2 days ago
df_correct['return_lag5'] = df_correct['return'].shift(5)   # Last week's return
df_correct['volatility_20d'] = df_correct['return'].shift(1).rolling(20).std()  # Trailing vol (shifted!)

# Drop NaN rows
df_correct = df_correct.dropna()

# Features: only lagged (past) information
X_correct = df_correct[['return_lag1', 'return_lag2', 'return_lag5', 'volatility_20d']]
y_correct = df_correct['target']

# Proper time-series split: train on past, test on future
train_size = int(len(df_correct) * 0.8)
X_train_correct = X_correct.iloc[:train_size]
X_test_correct = X_correct.iloc[train_size:]
y_train_correct = y_correct.iloc[:train_size]
y_test_correct = y_correct.iloc[train_size:]

# Train and evaluate
model_correct = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
model_correct.fit(X_train_correct, y_train_correct)
y_pred_correct = model_correct.predict(X_test_correct)

r2_correct = r2_score(y_test_correct, y_pred_correct)
print("=" * 60)
print("CORRECT APPROACH: Using only past data")
print("=" * 60)
print(f"R² Score: {r2_correct:.4f}")
print(f"\nThis is realistic for financial data - returns are hard to predict!")
print("An R² close to 0 (or even negative) means the model has no real predictive power.")

In [ ]:
# ============================================================
# THE WRONG WAY: Accidentally include future information
# ============================================================

# Common mistakes that cause leakage:

df_leaked = df_ts.copy()

# Mistake 1: Using a "centered" rolling window (looks into the future!)
# center=True means the window is centered on each point, using future data
df_leaked['vol_centered'] = df_leaked['return'].rolling(20, center=True).std()

# Mistake 2: Forgetting to shift when computing features
# This computes volatility using data up to AND INCLUDING today,
# but we're predicting TOMORROW - so this "knows" today which we shouldn't have
df_leaked['vol_wrong'] = df_leaked['return'].rolling(20).std()  # No shift!

# Mistake 3: The classic blunder - including the target (or something derived from it)
# "Oops, I wanted a smoothed version of returns for a feature..."
# This rolling mean INCLUDES the target day!
df_leaked['smooth_return'] = df_leaked['return'].rolling(3, center=True).mean()

# Worst case: directly including future return as a "lagged" feature (sign error)
df_leaked['return_lag1_WRONG'] = df_leaked['return'].shift(-1)  # This IS the target!

df_leaked = df_leaked.dropna()

# Features with leakage
X_leaked = df_leaked[['vol_centered', 'vol_wrong', 'smooth_return', 'return_lag1_WRONG']]
y_leaked = df_leaked['target']

# Same time-series split
train_size_leaked = int(len(df_leaked) * 0.8)
X_train_leaked = X_leaked.iloc[:train_size_leaked]
X_test_leaked = X_leaked.iloc[train_size_leaked:]
y_train_leaked = y_leaked.iloc[:train_size_leaked]
y_test_leaked = y_leaked.iloc[train_size_leaked:]

# Train and evaluate
model_leaked = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
model_leaked.fit(X_train_leaked, y_train_leaked)
y_pred_leaked = model_leaked.predict(X_test_leaked)

r2_leaked = r2_score(y_test_leaked, y_pred_leaked)
print("=" * 60)
print("WRONG APPROACH: Features include future information!")
print("=" * 60)
print(f"R² Score: {r2_leaked:.4f}")
print(f"\nWow, R² of {r2_leaked:.1%}! This model is AMAZING!")
print("...or is it? This is TOO GOOD TO BE TRUE.")
print("\nThe model 'works' because return_lag1_WRONG is literally the target!")

In [ ]:
# Compare side by side
print("=" * 60)
print("THE DANGER OF DATA LEAKAGE - SUMMARY")
print("=" * 60)
print(f"\n{'Approach':<30} {'R² Score':<15} {'Verdict'}")
print("-" * 60)
print(f"{'Correct (no leakage)':<30} {r2_correct:<15.4f} {'Realistic - near zero'}")
print(f"{'Wrong (with leakage)':<30} {r2_leaked:<15.4f} {'FAKE - way too good!'}")

print(f"\n⚠️  The leaked model has R² ≈ {r2_leaked:.0%} vs R² ≈ {r2_correct:.1%} for the correct model!")
print("\nIf you deployed the leaked model in production:")
print("  📈 Backtest: 'We can predict returns almost perfectly!'")
print("  📉 Live trading: Model performs like random guessing (or worse)")
print("  💸 Result: Massive losses, potentially career-ending")
print("\n" + "=" * 60)
print("ALWAYS ASK: 'At prediction time, would I actually have this data?'")
print("=" * 60)

In [ ]:
# Visualize the difference
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Correct model (left plot)
axes[0].scatter(y_test_correct, y_pred_correct, alpha=0.5, edgecolors='none', color='steelblue')
lims = [min(y_test_correct.min(), y_pred_correct.min()), 
        max(y_test_correct.max(), y_pred_correct.max())]
axes[0].plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Return')
axes[0].set_ylabel('Predicted Return')
axes[0].set_title(f'CORRECT: No Leakage (R² = {r2_correct:.4f})\nPredictions are basically noise')
axes[0].legend()

# Leaked model (right plot)
axes[1].scatter(y_test_leaked, y_pred_leaked, alpha=0.5, edgecolors='none', color='orange')
lims = [min(y_test_leaked.min(), y_pred_leaked.min()), 
        max(y_test_leaked.max(), y_pred_leaked.max())]
axes[1].plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
axes[1].set_xlabel('Actual Return')
axes[1].set_ylabel('Predicted Return')
axes[1].set_title(f'WRONG: With Leakage (R² = {r2_leaked:.4f})\nLooks amazing but is FAKE!')
axes[1].legend()

plt.tight_layout()
print("Left: Realistic results (can't predict returns well)")
print("Right: Too-good-to-be-true results (the model is cheating!)")

### How Pipelines Help Prevent Leakage

While pipelines can't prevent you from creating features with future data (that's on you!), they DO prevent a common form of leakage: **fitting transformers on test data**.

When you call `pipe.fit(X_train, y_train)`:
1. The scaler learns mean/std from `X_train` only
2. The model trains on scaled training data only

When you call `pipe.predict(X_test)`:
1. The scaler applies the **already-learned** mean/std to `X_test`
2. The model predicts on the scaled test data

**Key Rule**: Always fit on training data first, then transform/predict on test data.

---

### Red Flags That Suggest Leakage

Be suspicious when you see:

| Red Flag | Why It's Suspicious |
|----------|---------------------|
| **R² > 0.1 for daily returns** | In efficient markets, returns are nearly unpredictable |
| **Results much better than literature** | Published research R² for return prediction is typically < 0.05 |
| **"Centered" rolling windows** | `rolling(n, center=True)` uses future data! |
| **Features not shifted properly** | `df['vol'] = df['ret'].rolling(20).std()` includes today — need `.shift(1)` |
| **Sign errors in lag features** | `shift(-1)` looks forward, `shift(1)` looks backward |
| **Feature correlations > 0.5 with target** | Real predictive features rarely correlate this strongly with returns |

---

### The Golden Rule

> **At prediction time, could you actually have this feature in the real world?**

If it's Monday morning and you're predicting Monday's close:
- ✅ You CAN use: Friday's return, last week's volatility, last month's momentum
- ❌ You CANNOT use: Monday's return (that's what you're predicting!), Monday's volume, this week's average

---

## Part 5: Putting It All Together - A Complete Example

Let's build a complete pipeline for a factor model, demonstrating best practices.

In [ ]:
from sklearn.model_selection import cross_val_score

# Our features and target from earlier
print("Features:")
print(X.columns.tolist())
print(f"\nSamples: {len(X)}")

In [ ]:
# Create three pipelines with different models
pipelines = {
    'OLS': make_pipeline(StandardScaler(), LinearRegression()),
    'Ridge': make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    'Lasso': make_pipeline(StandardScaler(), Lasso(alpha=0.001)),
}

# Evaluate each using cross-validation
# We'll learn more about cross-validation in notebook 3!
results = []
for name, pipe in pipelines.items():
    scores = cross_val_score(pipe, X, y, cv=5, scoring='neg_mean_squared_error')
    # sklearn uses negative MSE (higher is better) for consistency
    mse_scores = -scores
    results.append({
        'Model': name,
        'Mean MSE': mse_scores.mean(),
        'Std MSE': mse_scores.std(),
        'MSE Range': f"{mse_scores.min():.5f} - {mse_scores.max():.5f}"
    })

results_df = pd.DataFrame(results)
print("5-Fold Cross-Validation Results:")
print(results_df.round(5).to_string(index=False))

In [ ]:
# Final model: Train on all data, inspect coefficients
final_model = pipelines['Ridge']
final_model.fit(X, y)

# Get coefficients (remember: these are for SCALED features)
coefs = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient (scaled)': final_model.named_steps['ridge'].coef_,
    'True Coefficient': true_coefficients
})

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
x_pos = np.arange(len(X.columns))
width = 0.35

bars1 = ax.bar(x_pos - width/2, coefs['Coefficient (scaled)'], width, label='Learned', color='steelblue')
bars2 = ax.bar(x_pos + width/2, coefs['True Coefficient'], width, label='True', color='coral')

ax.set_xlabel('Feature')
ax.set_ylabel('Coefficient')
ax.set_title('Learned vs True Coefficients (Note: Learned are on scaled features)')
ax.set_xticks(x_pos)
ax.set_xticklabels(X.columns, rotation=45, ha='right')
ax.legend()
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()

---

## Summary: Key Takeaways

### 1. The Three Interfaces
```python
# Estimator - learns from data
estimator.fit(X, y)

# Transformer - modifies data
X_new = transformer.transform(X)
X_new = transformer.fit_transform(X)  # fit + transform in one step

# Predictor - makes predictions
y_pred = predictor.predict(X)
```

### 2. Use Pipelines
```python
from sklearn.pipeline import make_pipeline

pipe = make_pipeline(
    StandardScaler(),
    Ridge(alpha=1.0)
)
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
```

### 3. Pipelines Prevent Data Leakage
- Always use pipelines in production code
- Never `fit_transform` on test data
- Never compute statistics on full data before splitting

### 4. The Same Pattern Works Everywhere
- Want to try a different model? Change one line
- Want to add preprocessing? Add a step to the pipeline
- Want to tune hyperparameters? Use GridSearchCV (coming in notebook 3)

---

## What's Next?

In the next notebook, we'll dive deep into **Transformers**:
- Different scalers and when to use each
- Encoding categorical variables
- Handling missing data
- **Time-series features**: Lags, rolling windows, momentum
- Building custom transformers

See you in `2_transformers.ipynb`!